In [0]:
from pyspark.sql.functions import  *

from pyspark.sql import  *

### DATA READING

In [0]:
df = spark.read.format('parquet').load('abfss://bronze@databricksretailproject.dfs.core.windows.net/orders')

In [0]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
df = df.withColumnRenamed('_rescued_data', 'rescued_data')

In [0]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- rescued_data: string (nullable = true)



In [0]:
df = df.drop('rescued_data')

In [0]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)



In [0]:
df = df.withColumn('order_date', to_timestamp(col('order_date')))

In [0]:
df = df.withColumn('year', year(col('order_date')))


In [0]:
df1 = df.withColumn('Dense_rank', dense_rank().over(Window.partitionBy('year').orderBy(desc('total_amount'))))

In [0]:
df1 = df.withColumn('Row_no', row_number().over(Window.partitionBy('year').orderBy(desc('total_amount'))))

### OOP CLASSES

In [0]:
class windows_Class:

    def dense_rank1(self, df):
        df_dense_rank = df.withColumn('Dense_rank', dense_rank().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
        return df_dense_rank

    def rank1(self, df):
        df_rank = df.withColumn('rank', rank().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
        return df_rank
    
    def Row_No1(self, df):
        df_Row_no = df.withColumn('row_No', row_number().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
        return df_Row_no

     

In [0]:
df_new = df

In [0]:
obj  = windows_Class()

In [0]:
df_new_result = obj.dense_rank1(df_new)


### DATA WRITING

In [0]:
df.write.format('delta').mode('append').save('abfss://silver@databricksretailproject.dfs.core.windows.net/orders')